In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction import DictVectorizer

from sklearn.pipeline import make_pipeline
#Metricas
from sklearn.metrics import (accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix)

import warnings
warnings.filterwarnings('ignore')

In [3]:
df = pd.read_csv('../data/diabetes_prediction_dataset_cleaned_v2.csv')

In [4]:
numerical = ['bmi', 'age', 'blood_glucose_level','HbA1c_level']
categorical = ['gender', 'hypertension', 'heart_disease', 'smoking_history']
features = ['gender','age','hypertension','heart_disease','smoking_history','bmi','HbA1c_level','blood_glucose_level']
TARGET = 'diabetes'

In [5]:
def prepare_features(df):
    """
    Convert a DataFrame into a list of dictionaries
    suitable for DictVectorizer.
    """

    df_features = df[features].copy()

    # Convert categorical features to strings
    for col in categorical:
        df_features[col] = df_features[col].astype(str)

    return df_features.to_dict(orient='records')

In [6]:
def split_data(df):
    """
    Split the dataset into train, validation and test sets.

    Proportions:
        Train: 60%
        Validation: 20%
        Test: 20%
    """

    # First split: 80% full train, 20% test
    df_full_train, df_test = train_test_split(
        df,
        test_size=0.20,
        random_state=42,
        stratify=df[TARGET]
    )

    # Second split: 75% train, 25% validation
    df_train, df_val = train_test_split(
        df_full_train,
        test_size=0.25,
        random_state=42,
        stratify=df_full_train[TARGET]
    )
    
    df_train = df_train.reset_index(drop=True)
    df_val = df_val.reset_index(drop=True)
    df_test = df_test.reset_index(drop=True)
    
    return df_full_train,df_train, df_val, df_test

In [7]:
models = {
    'LoR': LogisticRegression(
        max_iter=1000, class_weight='balanced',random_state=42),
    'dt': DecisionTreeClassifier(random_state=42),
    'rf': RandomForestClassifier(random_state=42),
}

In [8]:
def build_pipeline(model,**args):
  if isinstance(model,type):
    model = model(**args)
  else:
    model = model.set_params(**args)
  return make_pipeline(
      DictVectorizer(),
      model
  )

In [9]:
def train_model(model,df):
  X_train = prepare_features(df)
  y_train = df[TARGET].values
  pipeline = build_pipeline(model)
  pipeline.fit(X_train, y_train)
  return pipeline

In [10]:
def evaluate_model(model,df):

  X = prepare_features(df)
  y = df[TARGET].values

  y_pred = model.predict(X)
  y_prob = model.predict_proba(X)[:, 1]

  metrics = {'accuracy': float(accuracy_score(y, y_pred)),
             'precision': float(precision_score(y, y_pred, zero_division=0)),
             'recall': float(recall_score(y, y_pred, zero_division=0)),
             'f1_score': float(f1_score(y, y_pred, zero_division=0)),
             'roc_auc': float(roc_auc_score(y, y_prob)),
             'confusion_matrix': confusion_matrix(y, y_pred).tolist()
  }
  return metrics

In [11]:
df_full_train, df_train , df_val, df_test = split_data(df)
X_train , y_train = prepare_features(df_train), df[TARGET].values

In [ ]:

model_scores = []
for name, model in models.items():
    scores = []
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    
    # Usamos cross_validate en lugar de cross_val_score para soportar múltiples métricas
    for train_idx,val_idx in kfold.split(df_full_train):
        df_train = df_full_train.iloc[train_idx]
        df_val = df_full_train.iloc[val_idx]
        
        y_train , y_val = df_train[TARGET].values, df_val[TARGET].values
        model_pipeline = train_model(model,df_train)
        metrics = evaluate_model(model_pipeline,df_val) 
        scores.append(metrics['f1_score'])  
    model_scores.append(scores)      
       
        

In [14]:
for d in model_scores:
    d_array = np.array(d)
    print(d_array.mean())
    print(d_array.std())
    print()

0.5796765274716783
0.010076113821445177

0.7277316158045641
0.011387002311113417

0.79246163989833
0.017838342070797025



# Hyperparameter tunning

In [23]:
#Regresion logistica
from sklearn.model_selection import GridSearchCV

#Preparing for modelling
X_train = prepare_features(df_train)
y_train = df_train[TARGET].values

# definir el diccionario de hiperparametros
DictHP_log = {"logisticregression__C": np.logspace(-4,4,35),
              "logisticregression__solver" : ["lbfgs","liblinear"],
              "logisticregression__penalty" : ["l1","l2","elastic","none"]
              }

DictHP_dt = {
    'decisiontreeclassifier__max_depth' : [3,5,7],
    'decisiontreeclassifier__min_samples_leaf' : [1,3,5],
    'decisiontreeclassifier__ccp_alpha' : [0.0,0.01,0.05]
    }
DictHP_rf = {
    'randomforestclassifier__n_estimators':[10,20,30],
    'randomforestclassifier__max_depth':[5,10,15,20],
    'randomforestclassifier__min_samples_leaf': [3,5,7,9]
    }

In [16]:
models_param_grid = []

# Logistic Regression

In [17]:
#Configuramos el gridsearchCV
mod_base_Clf = build_pipeline(models['LoR'])
modClfGS1 = GridSearchCV(estimator = mod_base_Clf,
                         param_grid = DictHP_log,
                         cv = 5,
                         scoring = "f1",
                         n_jobs = -1,
                         verbose =4)

In [18]:
best_Clf_GS1 = modClfGS1.fit(X_train,y_train)

Fitting 5 folds for each of 280 candidates, totalling 1400 fits
[CV 1/5] END logisticregression__C=0.0001, logisticregression__penalty=l1, logisticregression__solver=lbfgs;, score=nan total time=   0.6s[CV 2/5] END logisticregression__C=0.0001, logisticregression__penalty=l1, logisticregression__solver=lbfgs;, score=nan total time=   0.6s

[CV 4/5] END logisticregression__C=0.0001, logisticregression__penalty=l1, logisticregression__solver=lbfgs;, score=nan total time=   0.7s
[CV 3/5] END logisticregression__C=0.0001, logisticregression__penalty=l1, logisticregression__solver=lbfgs;, score=nan total time=   0.8s
[CV 5/5] END logisticregression__C=0.0001, logisticregression__penalty=l1, logisticregression__solver=lbfgs;, score=nan total time=   0.7s
[CV 1/5] END logisticregression__C=0.0001, logisticregression__penalty=l1, logisticregression__solver=liblinear;, score=nan total time=   0.7s
[CV 2/5] END logisticregression__C=0.0001, logisticregression__penalty=l1, logisticregression__sol

In [19]:
best_Clf_GS1.best_params_

{'logisticregression__C': np.float64(0.11450475699382812),
 'logisticregression__penalty': 'l2',
 'logisticregression__solver': 'lbfgs'}

In [20]:
best_Clf_GS1.best_score_

np.float64(0.5801001057604473)

In [29]:
models_param_grid.append((best_Clf_GS1.best_score_,best_Clf_GS1.best_params_))

# Decision tree

In [24]:
mod2_base_Clf = build_pipeline(models['dt'])
modClfGS2 = GridSearchCV(estimator = mod2_base_Clf,
                         param_grid = DictHP_dt,
                         cv = 5,
                         scoring = "f1",
                         n_jobs = -1,
                         verbose =4)

In [25]:
best_Clf_GS2 = modClfGS2.fit(X_train,y_train)

Fitting 5 folds for each of 27 candidates, totalling 135 fits


[CV 1/5] END decisiontreeclassifier__ccp_alpha=0.0, decisiontreeclassifier__max_depth=3, decisiontreeclassifier__min_samples_leaf=1;, score=0.810 total time=   0.9s
[CV 2/5] END decisiontreeclassifier__ccp_alpha=0.0, decisiontreeclassifier__max_depth=3, decisiontreeclassifier__min_samples_leaf=1;, score=0.795 total time=   0.8s
[CV 3/5] END decisiontreeclassifier__ccp_alpha=0.0, decisiontreeclassifier__max_depth=3, decisiontreeclassifier__min_samples_leaf=1;, score=0.796 total time=   0.8s
[CV 4/5] END decisiontreeclassifier__ccp_alpha=0.0, decisiontreeclassifier__max_depth=3, decisiontreeclassifier__min_samples_leaf=1;, score=0.817 total time=   0.9s
[CV 5/5] END decisiontreeclassifier__ccp_alpha=0.0, decisiontreeclassifier__max_depth=3, decisiontreeclassifier__min_samples_leaf=1;, score=0.774 total time=   0.9s
[CV 1/5] END decisiontreeclassifier__ccp_alpha=0.0, decisiontreeclassifier__max_depth=3, decisiontreeclassifier__min_samples_leaf=3;, score=0.810 total time=   0.9s
[CV 2/5] E

In [30]:
models_param_grid.append((best_Clf_GS2.best_score_,best_Clf_GS2.best_params_))

# Random Forest

In [26]:
mod3_base_Clf = build_pipeline(models['rf'])
modClfGS3 = GridSearchCV(estimator = mod3_base_Clf,
                         param_grid = DictHP_rf,
                         cv = 5,
                         scoring = "f1",
                         n_jobs = -1,
                         verbose =4)

In [27]:
best_Clf_GS3 = modClfGS3.fit(X_train,y_train)


Fitting 5 folds for each of 48 candidates, totalling 240 fits
[CV 1/5] END randomforestclassifier__max_depth=5, randomforestclassifier__min_samples_leaf=3, randomforestclassifier__n_estimators=10;, score=0.769 total time=   1.3s
[CV 2/5] END randomforestclassifier__max_depth=5, randomforestclassifier__min_samples_leaf=3, randomforestclassifier__n_estimators=10;, score=0.775 total time=   1.3s
[CV 3/5] END randomforestclassifier__max_depth=5, randomforestclassifier__min_samples_leaf=3, randomforestclassifier__n_estimators=10;, score=0.783 total time=   1.2s
[CV 4/5] END randomforestclassifier__max_depth=5, randomforestclassifier__min_samples_leaf=3, randomforestclassifier__n_estimators=10;, score=0.761 total time=   1.1s
[CV 5/5] END randomforestclassifier__max_depth=5, randomforestclassifier__min_samples_leaf=3, randomforestclassifier__n_estimators=10;, score=0.759 total time=   1.3s
[CV 1/5] END randomforestclassifier__max_depth=5, randomforestclassifier__min_samples_leaf=3, randomfor

In [31]:
models_param_grid.append((best_Clf_GS3.best_score_,best_Clf_GS3.best_params_))

In [ ]:
pd.DataFrame(models_param_grid,columns=['score f1','params'])

,score f1,params
0,0.58010,"GridSearchCV(cv=5,\n estimator=Pip..."
1,0.58010,"{'logisticregression__C': 0.11450475699382812,..."
2,0.79865,"{'decisiontreeclassifier__ccp_alpha': 0.0, 'de..."
3,0.80123,"{'randomforestclassifier__max_depth': 15, 'ran..."


In [35]:
best_Clf_GS3.best_params_

{'randomforestclassifier__max_depth': 15,
 'randomforestclassifier__min_samples_leaf': 5,
 'randomforestclassifier__n_estimators': 10}